# 00 · Foundation

The shared pieces every agent uses. Pablo set up the contracts (config, schemas, state,
memory); this notebook adds one LLM client for all agents.

| Section | Module | What it is |
|---|---|---|
| Config | `src/config.py` | `config.yaml` loader, repo paths, API keys from `.env` |
| Schemas | `src/schemas.py` | `CardListing`, `ParsedQuery`, `EvaluatedListing`, ... |
| State | `src/state.py` | `AgentState`, the typed graph state all agents read and write |
| Memory | `src/memory.py` | SQLite buyer profile and query history |
| LLM | `src/llm.py` | Gemini/Groq client: cache, retries, free-tier fallback, offline mode |

Each `#| export` cell is copied into its module by the last cell of this notebook.
Run order for the whole project: 00 → 01 → ... → 07.

In [ ]:
#| default_exp config

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

## Config

Every number lives in `config.yaml`; code never hardcodes thresholds or model ids.

In [ ]:
#| export
"""Configuration loader for config.yaml, plus repo paths and API keys from .env."""

import os
from functools import cache
from pathlib import Path
from typing import Any

import yaml
from dotenv import load_dotenv

In [ ]:
#| export
def find_root() -> Path:
    """The repo root: the first folder upward with config.yaml in it, starting from this
    module (or, inside a notebook, from the working directory)."""
    start = Path(globals().get("__file__", Path.cwd())).resolve()
    for folder in [start, *start.parents]:
        if (folder / "config.yaml").exists():
            return folder
    raise FileNotFoundError("config.yaml not found; run from inside the repo")

In [ ]:
#| export
ROOT = find_root()
DEFAULT_CONFIG_PATH = ROOT / "config.yaml"

In [ ]:
#| export
def load_config(path: Path | str = DEFAULT_CONFIG_PATH) -> dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)

In [ ]:
#| export
def repo_path(rel: str | Path) -> Path:
    """Absolute path for a path written relative to the repo root, as in config.yaml."""
    p = Path(rel)
    return p if p.is_absolute() else ROOT / p

In [ ]:
#| export
@cache
def _load_dotenv() -> None:
    load_dotenv(ROOT / ".env")

In [ ]:
#| export
def env(*names: str) -> str | None:
    """First non-empty environment variable among `names`. Reads .env once."""
    _load_dotenv()
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return None

## Schemas

The shared data contracts. Additions to Pablo's version, all optional with defaults:
`CardListing.set_name` (spoken in replies) and `card_key`; `ParsedQuery.variant`,
`any_printing`, `asked_price` (price checks) and `filled_from_memory`.

In [ ]:
#| export schemas
"""Shared Pydantic schemas for listings, queries, and evaluations."""

from typing import Literal

from pydantic import BaseModel, Field

In [ ]:
#| export schemas
Condition = Literal["NM", "LP", "MP", "HP", "DMG", "unknown"]
Variant = Literal["normal", "holo", "reverse", "1st edition", "unknown"]
DeliveryType = Literal["in_person", "shipping", "unknown"]
Persona = Literal["budget_parent", "serious_collector", "cautious_buyer"]

CONDITION_ORDER = ["DMG", "HP", "MP", "LP", "NM"]  # worst to best
CONDITION_WORDS = {
    "NM": "near mint",
    "LP": "lightly played",
    "MP": "moderately played",
    "HP": "heavily played",
    "DMG": "damaged",
    "unknown": "unstated",
}

In [ ]:
#| export schemas
class CardListing(BaseModel):
    """Anonymized, structured single-card listing schema (Section 4 & 14)."""

    listing_id: str
    raw_title: str
    raw_description: str = ""
    price: float = Field(ge=0.0)
    location_city: str = ""
    delivery_type: DeliveryType = "unknown"
    has_clear_photo: bool = True

    # Extracted card identity fields
    card_name: str = "unknown"
    set_name: str | None = None  # as written or implied, e.g. "Base Set"; spoken in replies
    set_code: str = "unknown"  # TCGdex set id, e.g. "base1"
    number: str = "unknown"
    variant: Variant = "unknown"
    condition: Condition = "unknown"
    graded: bool = False
    grade: str | None = None
    language: str = "en"
    is_lot: bool = False
    shipping: float | None = Field(default=None, ge=0.0)
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)

    # Listing signals for the trust and convenience scores (None: not known, e.g. samples)
    photo_count: int | None = None
    listed_at: str | None = None  # date the listing was posted, "YYYY-MM-DD"
    distance_miles: float | None = None  # from the buyer's city (config `buyer`)
    public_meetup: bool = False  # the seller offers to meet in public
    pending: bool = False  # Facebook marks the item as pending a sale

    # Evaluation metadata (traps & decoys)
    synthetic: bool = False
    split: Literal["dev", "test"] | None = None
    label: Literal["clean", "trap", "decoy"] | None = None
    label_reason: str | None = None

    @property
    def card_key(self) -> str | None:
        """Price key "<set_code>-<number>", e.g. "base1-4", or None if either is unknown."""
        if self.set_code == "unknown" or self.number == "unknown":
            return None
        return f"{self.set_code}-{self.number}"

In [ ]:
#| export schemas
class ParsedQuery(BaseModel):
    """Structured buyer request produced by the Consumer Agent in INTENT."""

    card_name: str | None = None
    set_code: str | None = None  # TCGdex set id or set name, matched loosely by the Merchant
    budget: float | None = Field(default=None, gt=0.0)
    min_condition: Condition | None = None
    variant: Variant | None = None
    excluded_variants: list[str] = Field(default_factory=list)
    any_printing: bool = False  # "any Charizard is fine": skip the printing question
    persona: Persona = "cautious_buyer"
    is_consultative: bool = False  # e.g. "is $40 fair for a Base Set Charizard?"
    asked_price: float | None = Field(default=None, ge=0.0)  # the price a price check asks about
    filled_from_memory: list[str] = Field(default_factory=list)  # fields taken from the profile

In [ ]:
#| export schemas
class LandedCostResult(BaseModel):
    """Output of the deterministic landed cost tool."""

    price: float
    shipping: float
    tax: float
    total: float
    shipping_estimated: bool

In [ ]:
#| export schemas
class EvaluatedListing(BaseModel):
    """Listing enriched with Merchant reflection and Consumer scoring."""

    listing: CardListing
    market_price: float | None = None  # TCGplayer market price for the printing and variant
    risk_score: float = Field(default=0.0, ge=0.0, le=1.0)
    risk_reasons: list[str] = Field(default_factory=list)
    warning: str | None = None
    landed_cost: LandedCostResult | None = None
    utility_score: float | None = None  # 0-1; the spoken "score" is this times 100
    score_parts: dict[str, float] = Field(default_factory=dict)  # each part of the score, 0-1
    reasons: list[str] = Field(default_factory=list)  # why it scored well, in spoken words
    rejected: bool = False
    rejection_reason: str | None = None

## State

Added: `price_check`, `live_search` and `response_source`.

In [ ]:
#| export state
"""Shared LangGraph state interface and tool signatures (Section 3 & 10)."""

from typing import Any, Literal, TypedDict

from src.schemas import CardListing, EvaluatedListing, ParsedQuery

In [ ]:
#| export state
GraphStage = Literal[
    "INTENT",
    "CLARIFY",
    "CONFIRM",
    "SEARCHING",
    "EVALUATING",
    "RANKING",
    "RESPONDING",
    "DONE",
]

In [ ]:
#| export state
class AgentState(TypedDict, total=False):
    """Typed state shared across Platform, Consumer, Merchant, and Humanizer agents."""

    # Input & session tracking
    session_id: str
    user_input: str
    current_state: GraphStage

    # Consumer Agent (INTENT)
    parsed_query: ParsedQuery | None

    # Platform Agent (CLARIFY)
    clarify_count: int
    clarifying_question: str | None
    clarify_answer: str | None

    # Platform Agent (CONFIRM): the request read back before searching
    confirm_first: bool  # set by ChatSession: read the request back before searching
    confirm_count: int
    confirm_question: str | None
    confirmed: bool  # the buyer said yes: the next run starts at SEARCHING

    # Merchant Agent (SEARCHING & EVALUATING)
    search_retry_count: int
    relaxed_filters: bool
    candidate_listings: list[CardListing]
    evaluated_listings: list[EvaluatedListing]
    reflection_complete: bool
    price_check: dict[str, Any] | None  # market facts for a consultative price question
    live_search: dict[str, Any] | None  # summary of a live Marketplace search, if one ran

    # Platform & Consumer Agents (Policy & RANKING)
    ranked_listings: list[EvaluatedListing]
    rejected_listings: list[EvaluatedListing]
    closest_rejected: EvaluatedListing | None

    # Humanizer Agent (RESPONDING)
    humanizer_retry_count: int
    draft_response: str | None
    number_check_passed: bool
    number_check_error: str | None
    final_response: str | None
    response_source: str | None  # "llm" or "template"
    sample_note_said: bool  # set by ChatSession: the sample-data note was already given

    # Telemetry & logging
    stage_timings: dict[str, float]
    token_usage: dict[str, int]

## Memory

Pablo's SQLite store, plus `reset()` for demos.

In [ ]:
#| export memory
"""SQLite buyer profile and query history memory (Section 3, 7, & 11)."""

import json
import sqlite3
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from src.config import ROOT

In [ ]:
#| export memory
DEFAULT_DB_PATH = ROOT / "data" / "buyer_memory.sqlite"

In [ ]:
#| export memory
class BuyerMemory:
    """Local SQLite store for buyer budget, persona, preferred sets/conditions, and past queries."""

    def __init__(self, db_path: Path | str = DEFAULT_DB_PATH) -> None:
        self.db_path = Path(db_path)
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self._init_db()

    def _connect(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_db(self) -> None:
        with self._connect() as conn:
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS buyer_profiles (
                    session_id TEXT PRIMARY KEY,
                    budget REAL,
                    persona TEXT,
                    preferred_sets TEXT,
                    preferred_condition TEXT,
                    updated_at TEXT
                )
                """
            )
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS query_history (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    raw_query TEXT NOT NULL,
                    parsed_json TEXT,
                    top_pick_id TEXT,
                    created_at TEXT NOT NULL
                )
                """
            )

    def get_profile(self, session_id: str) -> dict[str, Any]:
        """Load saved buyer defaults for a session."""
        with self._connect() as conn:
            row = conn.execute(
                "SELECT * FROM buyer_profiles WHERE session_id = ?",
                (session_id,),
            ).fetchone()
        if row is None:
            return {}
        return {
            "session_id": row["session_id"],
            "budget": row["budget"],
            "persona": row["persona"],
            "preferred_sets": json.loads(row["preferred_sets"] or "[]"),
            "preferred_condition": row["preferred_condition"],
            "updated_at": row["updated_at"],
        }

    def save_profile(
        self,
        session_id: str,
        budget: float | None = None,
        persona: str | None = None,
        preferred_set: str | None = None,
        preferred_condition: str | None = None,
    ) -> dict[str, Any]:
        """Upsert buyer preferences, preserving previously saved values when new ones are None."""
        existing = self.get_profile(session_id)
        merged_budget = budget if budget is not None else existing.get("budget")
        merged_persona = persona if persona is not None else existing.get("persona")
        merged_condition = (
            preferred_condition
            if preferred_condition is not None
            else existing.get("preferred_condition")
        )
        sets: list[str] = list(existing.get("preferred_sets", []))
        if preferred_set and preferred_set not in sets:
            sets.append(preferred_set)

        now = datetime.now(UTC).isoformat()
        with self._connect() as conn:
            conn.execute(
                """
                INSERT INTO buyer_profiles (
                    session_id, budget, persona, preferred_sets, preferred_condition, updated_at
                )
                VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(session_id) DO UPDATE SET
                    budget = excluded.budget,
                    persona = excluded.persona,
                    preferred_sets = excluded.preferred_sets,
                    preferred_condition = excluded.preferred_condition,
                    updated_at = excluded.updated_at
                """,
                (
                    session_id,
                    merged_budget,
                    merged_persona,
                    json.dumps(sets),
                    merged_condition,
                    now,
                ),
            )
        return self.get_profile(session_id)

    def log_query(
        self,
        session_id: str,
        raw_query: str,
        parsed_dict: dict[str, Any] | None = None,
        top_pick_id: str | None = None,
    ) -> None:
        """Append a completed query to history."""
        now = datetime.now(UTC).isoformat()
        with self._connect() as conn:
            conn.execute(
                """
                INSERT INTO query_history (session_id, raw_query, parsed_json, top_pick_id, created_at)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    session_id,
                    raw_query,
                    json.dumps(parsed_dict) if parsed_dict is not None else None,
                    top_pick_id,
                    now,
                ),
            )

    def get_history(self, session_id: str, limit: int = 10) -> list[dict[str, Any]]:
        """Return most recent queries for a session."""
        with self._connect() as conn:
            rows = conn.execute(
                """
                SELECT raw_query, parsed_json, top_pick_id, created_at
                FROM query_history
                WHERE session_id = ?
                ORDER BY id DESC
                LIMIT ?
                """,
                (session_id, limit),
            ).fetchall()
        return [
            {
                "raw_query": r["raw_query"],
                "parsed": json.loads(r["parsed_json"]) if r["parsed_json"] else None,
                "top_pick_id": r["top_pick_id"],
                "created_at": r["created_at"],
            }
            for r in rows
        ]

    def reset(self, session_id: str) -> None:
        """Forget a session's profile and history (used by the notebooks' demos)."""
        with self._connect() as conn:
            conn.execute("DELETE FROM buyer_profiles WHERE session_id = ?", (session_id,))
            conn.execute("DELETE FROM query_history WHERE session_id = ?", (session_id,))

## LLM

One client for every agent, so caching, retries and the model choice live in one place.

- **Model:** `llm.active` in `config.yaml` (`primary` = Gemini 3.8 Flash). The env variable
  `AGENTIC_LLM` overrides it; the setup cell above sets it from `LIVE`.
- **Cache:** every response is stored in `.cache/llm`, so re-running a cell is free.
- **Rate limits:** a 429 is retried with backoff; if the quota is gone, the call moves to
  `llm.fallback` (`gemini-3.5-flash-lite`, higher free daily quota).
- **Offline:** `AGENTIC_LLM=offline` answers each call tag with a rule-based stand-in that
  the agent registers with `@offline_handler`. No key, no cost; the tests use it.

In [ ]:
#| export llm
"""One LLM client for every agent: Gemini or Groq, a disk cache, retries on rate limits, an
automatic free-tier fallback, and an offline mode that answers with rules instead of an API."""

import hashlib
import json
import os
import time
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any, TypeVar

from diskcache import Cache
from pydantic import BaseModel

from src.config import env, repo_path

In [ ]:
#| export llm
T = TypeVar("T", bound=BaseModel)

# Offline stand-ins, one per call tag ("intent", "extract", ...). Each agent registers its own
# when its module is imported, so offline mode needs no API key and costs nothing.
OFFLINE_HANDLERS: dict[str, Callable[[str, str], Any]] = {}

In [ ]:
#| export llm
def offline_handler(tag: str):
    """Decorator: register `fn(system, prompt)` as the offline answer for calls with `tag`."""

    def register(fn: Callable[[str, str], Any]) -> Callable[[str, str], Any]:
        OFFLINE_HANDLERS[tag] = fn
        return fn

    return register

In [ ]:
#| export llm
class LLMError(RuntimeError):
    pass

In [ ]:
#| export llm
@dataclass
class CallLog:
    tag: str
    model: str
    cached: bool
    seconds: float
    prompt_tokens: int = 0
    completion_tokens: int = 0

In [ ]:
#| export llm
def provider_for(model: str) -> str:
    """gemini-* -> gemini, rules -> offline, anything else (e.g. openai/gpt-oss-20b) -> groq."""
    if model == "rules":
        return "offline"
    return "gemini" if model.startswith("gemini") else "groq"

In [ ]:
#| export llm
def _rate_limited(e: Exception) -> bool:
    text = str(e)
    code = getattr(e, "code", None) or getattr(e, "status_code", None)
    return code == 429 or "429" in text or "RESOURCE_EXHAUSTED" in text

In [ ]:
#| export llm
@dataclass
class LLM:
    model: str
    temperature: float = 0.0
    max_retries: int = 3
    cache_dir: str | None = ".cache/llm"
    repeat: int = 0  # part of the cache key, so experiment repeats make real calls
    thinking_level: str | None = None  # minimal, low, medium, high (None = model default)
    fallback: LLM | None = None  # used once the quota for `model` runs out
    calls: list[CallLog] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.provider = provider_for(self.model)
        use_cache = self.cache_dir and self.provider != "offline"
        self._cache = Cache(str(repo_path(self.cache_dir))) if use_cache else None
        self._client = None

    def json(self, *, tag: str, system: str, prompt: str, schema: type[T]) -> T:
        """Structured output, validated against a Pydantic schema."""
        raw = self._call(tag, system, prompt, schema)
        if isinstance(raw, schema):
            return raw
        return schema.model_validate(raw if isinstance(raw, dict) else json.loads(raw))

    def text(self, *, tag: str, system: str, prompt: str) -> str:
        return str(self._call(tag, system, prompt, None)).strip()

    def usage(self) -> dict[str, int]:
        """Token totals over every call so far, including the fallback model's."""
        logs = self.calls + (self.fallback.calls if self.fallback else [])
        return {
            "calls": len(logs),
            "cached_calls": sum(c.cached for c in logs),
            "prompt_tokens": sum(c.prompt_tokens for c in logs),
            "completion_tokens": sum(c.completion_tokens for c in logs),
        }

    def _key(self, system: str, prompt: str, schema: type[BaseModel] | None) -> str:
        blob = json.dumps(
            [self.model, system, prompt, schema.__name__ if schema else None, self.temperature,
             self.repeat, self.thinking_level],
            ensure_ascii=False,
        )  # fmt: skip
        return hashlib.sha256(blob.encode()).hexdigest()

    def _call(self, tag: str, system: str, prompt: str, schema: type[BaseModel] | None):
        start = time.perf_counter()
        if self.provider == "offline":
            handler = OFFLINE_HANDLERS.get(tag)
            if handler is None:
                raise LLMError(f"Offline mode has no handler for '{tag}'")
            out = handler(system, prompt)
            self.calls.append(CallLog(tag, self.model, False, time.perf_counter() - start))
            return out

        key = self._key(system, prompt, schema)
        if self._cache is not None and key in self._cache:
            self.calls.append(CallLog(tag, self.model, True, 0.0))
            return self._cache[key]
        try:
            out, tokens = self._with_retries(system, prompt, schema)
        except LLMError as e:
            if self.fallback is not None and "quota" in str(e):
                print(f"[llm] {self.model} is out of quota; using {self.fallback.model}")
                return self.fallback._call(tag, system, prompt, schema)
            raise
        self.calls.append(CallLog(tag, self.model, False, time.perf_counter() - start, *tokens))
        if self._cache is not None:
            self._cache[key] = out
        return out

    def _with_retries(self, system, prompt, schema):
        delay = 5.0
        for attempt in range(self.max_retries + 1):
            try:
                if self.provider == "gemini":
                    return self._gemini(system, prompt, schema)
                return self._groq(system, prompt, schema)
            except LLMError:
                raise
            except Exception as e:
                if not _rate_limited(e):
                    raise LLMError(f"{self.model} call failed: {e}") from e
                if attempt == self.max_retries:
                    raise LLMError(f"{self.model}: rate limit or daily quota reached") from e
                time.sleep(delay)
                delay *= 2
        raise LLMError("unreachable")

    def _gemini(self, system, prompt, schema):
        from google import genai
        from google.genai import types

        if self._client is None:
            key = env("GEMINI_API_KEY")
            if not key:
                raise LLMError("GEMINI_API_KEY is missing. Add it to .env (see .env.example).")
            self._client = genai.Client(api_key=key)
        cfg: dict[str, Any] = {"system_instruction": system, "temperature": self.temperature}
        if self.thinking_level:
            cfg["thinking_config"] = types.ThinkingConfig(
                thinking_level=self.thinking_level.upper()
            )
        if schema is not None:
            cfg["response_mime_type"] = "application/json"
            cfg["response_schema"] = schema
        resp = self._client.models.generate_content(
            model=self.model, contents=prompt, config=types.GenerateContentConfig(**cfg)
        )
        meta = resp.usage_metadata
        # Thinking tokens are billed as output, so they count as completion tokens here.
        out_tokens = (
            (meta.candidates_token_count or 0) + (meta.thoughts_token_count or 0) if meta else 0
        )
        tokens = ((meta.prompt_token_count or 0) if meta else 0, out_tokens)
        if schema is None:
            return resp.text or "", tokens
        parsed = resp.parsed
        out = parsed.model_dump() if isinstance(parsed, BaseModel) else json.loads(resp.text)
        return out, tokens

    def _groq(self, system, prompt, schema):
        from groq import Groq

        if self._client is None:
            key = env("GROQ_API_KEY")
            if not key:
                raise LLMError("GROQ_API_KEY is missing. Add it to .env (see .env.example).")
            self._client = Groq(api_key=key)
        kwargs: dict[str, Any] = {}
        if self.thinking_level and "gpt-oss" in self.model:  # gpt-oss: low | medium | high
            kwargs["reasoning_effort"] = {"minimal": "low"}.get(
                self.thinking_level, self.thinking_level
            )
        if schema is not None:
            system += (
                "\n\nReply with only a JSON object that matches this JSON Schema:\n"
                + json.dumps(schema.model_json_schema())
            )
            kwargs["response_format"] = {"type": "json_object"}
        resp = self._client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            **kwargs,
        )
        text = resp.choices[0].message.content or ""
        u = resp.usage
        tokens = (u.prompt_tokens or 0, u.completion_tokens or 0) if u else (0, 0)
        return (json.loads(text) if schema is not None else text), tokens

In [ ]:
#| export llm
def make_llm(cfg: dict[str, Any], model_key: str | None = None, repeat: int = 0) -> LLM:
    """The LLM named by `model_key`, else env AGENTIC_LLM, else config `llm.active`.

    `model_key` is a key under `models:` in config.yaml, e.g. "primary", "small" or "offline".
    """
    key = model_key or os.getenv("AGENTIC_LLM") or cfg["llm"]["active"]
    settings = cfg["llm"]

    def build(k: str) -> LLM:
        return LLM(
            model=cfg["models"][k],
            temperature=cfg["models"].get("temperature", 0.0),
            max_retries=settings["max_retries"],
            cache_dir=settings["cache_dir"],
            repeat=repeat,
            thinking_level=settings.get("thinking_level"),
        )

    llm = build(key)
    fb = settings.get("fallback")
    if fb and fb != key and llm.provider == "gemini":
        llm.fallback = build(fb)
    return llm

# Try it

In [ ]:
import tempfile

from src.config import env, load_config
from src.llm import make_llm
from src.memory import BuyerMemory
from src.schemas import ParsedQuery

cfg = load_config()
print("primary model:", cfg["models"]["primary"], "| fallback:", cfg["models"]["primary_free_tier"])
print(
    "keys in .env:",
    {k: bool(env(k)) for k in ["GEMINI_API_KEY", "GROQ_API_KEY", "APIFY_API_TOKEN", "APIFY_TOKEN"]},
)

In [ ]:
q = ParsedQuery(card_name="Charizard", set_code="base1", budget=60, persona="serious_collector")
q.model_dump()

In [ ]:
mem = BuyerMemory(Path(tempfile.mkdtemp()) / "demo.sqlite")
mem.save_profile("demo", budget=60, persona="serious_collector")
mem.save_profile("demo", preferred_set="base1")  # keeps the budget and persona
mem.get_profile("demo")

With `LIVE = True` this makes one real (cached) Gemini call.

In [ ]:
llm = make_llm(cfg)
print("using", llm.model, "via", llm.provider)
if LIVE:
    print(
        llm.text(tag="ping", system="Reply in five words.", prompt="Say hello to a card collector.")
    )
    print(llm.usage())

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()